# Market Basket Analysis: Frequent Itemsets and Sequential Patterns

**CSCE 676 — Data Mining and Analysis**
**Texas A&M University — Spring 2026**
**Author:** Ahad Hussain (UIN 535007416)


## 1. Introduction & Motivation

Every day, millions of grocery baskets move through online platforms like Instacart. Each
basket is a small window into how people plan meals, manage households, and substitute
products. Across millions of baskets, stable patterns emerge: items that co-occur far more
often than chance would predict, purchase sequences that unfold across visits, and shopper
archetypes that look quite different from one another.

This project mines those patterns from the **Instacart Online Grocery Shopping Dataset 2017**
(~3.4M orders, ~32M order-product lines, ~49K products, ~206K users) and asks four layered
questions:

1. **RQ1 — Support sensitivity.** What association rules exist in the data, and how does
   the choice of minimum support threshold change what we see?
2. **RQ2 — Temporal context.** Do weekend and weekday shoppers — or morning and evening
   shoppers — produce different rules?
3. **RQ3 — Temporal order.** When we treat users' orders as *sequences* rather than bags,
   do we uncover progressions that unordered mining cannot see?
4. **RQ4 — Shopper type.** If we cluster users by what they buy, do different shopper
   archetypes produce different rules?

Each question is a different lens on the same underlying baskets. Together they build an
argument that **one-size-fits-all rule mining leaves a lot of structure on the table** —
and that segmenting by time, order, and user type all surface patterns that global mining
misses.


### 1.1 RQ-to-Method Mapping

| RQ  | Technique                                     | Algorithm            | Course / Beyond       | Key Metrics                                          |
|-----|-----------------------------------------------|----------------------|-----------------------|------------------------------------------------------|
| RQ1 | Frequent itemsets + association rules         | FP-Growth            | Course                | Support, confidence, lift, item diversity, runtime   |
| RQ2 | Segmented frequent itemsets                   | FP-Growth / segment  | Course                | Jaccard of rule sets, segment-exclusive rules, lift  |
| RQ3 | Sequential pattern mining                     | PrefixSpan           | **Beyond course**     | Sequential support, sequential-only patterns         |
| RQ4 | User clustering + cluster-specific rules      | K-Means + FP-Growth  | Course (clustering)   | Silhouette, cluster profiles, cluster-exclusive rules|


### 1.2 Collaboration Declaration

- **Human collaborators:** None. This is an individual project.
- **External resources:** Instacart 2017 public dataset (Kaggle); `mlxtend`, `prefixspan`,
  and `scikit-learn` library documentation; standard references (see end of notebook).
- **AI tools:** Anthropic Claude and OpenAI ChatGPT were used to sanity-check plan
  structure and scaffold boilerplate text. All analytical decisions, parameter choices,
  result interpretation, and code verification are my own.


## 2. Setup & Data Loading

We load the full `order_products__prior` file (~32M rows) and merge it with product, aisle,
and department metadata. Two basket representations are then built:

1. **Department-level baskets** (21 categories) — small enough to run on the **full**
   ~3.2M-order dataset as a dense boolean matrix.
2. **Product-level baskets** (~49K products) — far too large for a dense matrix
   (500K × 49K ≈ 24 GB), so we use `mlxtend`'s **sparse** `TransactionEncoder` on a
   500K-order random sample.

We also build two derived artifacts used in later sections:

- **User × department frequency matrix** (for RQ4 clustering)
- **Per-user dominant-department sequences** (for RQ3 PrefixSpan)


In [ ]:
# Core imports
import os
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

def md(text): display(Markdown(text))
def md_table(df, title=None, n=None):
    if title: display(Markdown(f"**{title}**"))
    display(df.head(n) if n else df)

# --- Progress logging helpers -------------------------------------------
# Long-running cells can be silent for many minutes in Colab. `log()` emits
# timestamped lines with flush=True so Colab's stdout buffer shows progress
# live, and `Timer` pairs a START / DONE around any block so the user can
# see which step is currently executing and how long each one took.
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

class Timer:
    """Context manager that logs entry + exit with elapsed seconds."""
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.time()
        log(f"START {self.label}")
        return self
    def __exit__(self, *exc):
        log(f"DONE  {self.label} in {time.time() - self.t0:.1f}s")

DATA_DIR = Path("data/instacart")
assert DATA_DIR.exists(), f"Data directory missing: {DATA_DIR.resolve()}"
print("Data dir:", DATA_DIR.resolve())


In [ ]:
# --- Config / scale knobs -----------------------------------------------
# Set FAST_MODE=True to run everything on a small sample (~5 min end-to-end).
# Set FAST_MODE=False for the full deliverable run.
FAST_MODE = False

PRODUCT_SAMPLE_N = 100_000 if FAST_MODE else 500_000   # orders for product-level FP-Growth
PREFIXSPAN_USER_N = 20_000 if FAST_MODE else 50_000    # users for PrefixSpan
RQ1_THRESHOLDS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
RQ2_SUPPORT = 0.01
RQ4_K_RANGE = range(2, 9)

# --- Parallelism --------------------------------------------------------
# Independent loops in this notebook (RQ1 threshold sweep, RQ4 KMeans
# k-sweep, per-cluster FP-Growth) are dispatched with joblib so they use
# every available core. N_JOBS_HEAVY caps parallelism for the product-level
# sweep because the 500K x 46K sparse matrix is pickled once per worker and
# low-support runs allocate several GB each.
N_JOBS = max(1, os.cpu_count() or 1)
N_JOBS_HEAVY = min(N_JOBS, 2)

print(f"FAST_MODE        = {FAST_MODE}")
print(f"N_JOBS           = {N_JOBS} (available cores: {os.cpu_count()})")
print(f"N_JOBS_HEAVY     = {N_JOBS_HEAVY}  (product-level sweep)")
print(f"Product sample   = {PRODUCT_SAMPLE_N:,} orders")
print(f"PrefixSpan users = {PREFIXSPAN_USER_N:,}")


In [ ]:
# --- Load raw tables ----------------------------------------------------
with Timer("loading parquet tables"):
    orders = pd.read_parquet(DATA_DIR / "orders.parquet")
    order_products_prior = pd.read_parquet(DATA_DIR / "order_products__prior.parquet")
    products = pd.read_parquet(DATA_DIR / "products.parquet")
    aisles = pd.read_parquet(DATA_DIR / "aisles.parquet")
    departments = pd.read_parquet(DATA_DIR / "departments.parquet")
print(f"  orders:                {len(orders):>12,}")
print(f"  order_products_prior:  {len(order_products_prior):>12,}")
print(f"  products:              {len(products):>12,}")
print(f"  aisles:                {len(aisles):>12,}")
print(f"  departments:           {len(departments):>12,}")


In [4]:
# --- Enrich products with aisle and department names -------------------
products_enriched = (
    products
      .merge(aisles, on="aisle_id")
      .merge(departments, on="department_id")
)
products_enriched.head(3)


,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages


In [ ]:
# --- Merge order lines with product metadata + order temporal features --
with Timer("merging order_lines with product + order metadata"):
    op = order_products_prior.merge(
        products_enriched[["product_id", "product_name", "aisle", "department"]],
        on="product_id"
    )
    op = op.merge(
        orders[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day"]],
        on="order_id"
    )
print(f"  op shape: {op.shape}")
op.head(3)


### 2.1 Build Department-Level Baskets (Full Dataset)

Department-level baskets are small enough to run on the full ~3.2M-order prior dataset.


In [ ]:
# Fast path for building dept baskets:
# the previous version did `groupby.apply(lambda s: sorted(set(s)))` which pays
# Python-lambda overhead once per group (3.2M groups => ~40s). We instead
# `drop_duplicates` so each (order_id, department) pair appears once, then do
# a single `.agg(list)` which pandas can execute in Cython.
with Timer("building department-level baskets (full ~3.2M orders)"):
    dept_pairs = op[["order_id", "department"]].drop_duplicates()
    log(f"  unique (order_id, department) pairs: {len(dept_pairs):,}")
    dept_baskets_series = (
        dept_pairs.sort_values(["order_id", "department"])
                  .groupby("order_id", sort=False)["department"]
                  .agg(list)
    )
    dept_baskets = dept_baskets_series.tolist()
    log(f"  built {len(dept_baskets):,} dept baskets")

with Timer("encoding dept basket matrix"):
    te_dept = TransactionEncoder()
    dept_basket_matrix = te_dept.fit_transform(dept_baskets)
    # Index by order_id so downstream segmentation / clustering can do
    # `dept_basket_df.loc[order_ids]` instead of rebuilding from `op` each call.
    dept_basket_df = pd.DataFrame(
        dept_basket_matrix, columns=te_dept.columns_,
        index=dept_baskets_series.index,
    )
    dept_basket_df.index.name = "order_id"
    log(f"  matrix: {dept_basket_df.shape} "
        f"(~{dept_basket_df.memory_usage(deep=True).sum()/1e6:.0f} MB)")


### 2.2 Build Product-Level Baskets (500K-order Sample, Sparse)

Dense encoding of 500K × 49K booleans would require ~24 GB. We use `mlxtend`'s
`sparse=True` path, which stores baskets in SciPy CSR format (roughly one byte per
item-in-basket rather than one per cell).


In [ ]:
rng = np.random.default_rng(42)
sample_order_ids = rng.choice(
    op["order_id"].unique(), size=PRODUCT_SAMPLE_N, replace=False
)
sample_order_ids_set = set(sample_order_ids.tolist())
op_sample = op[op["order_id"].isin(sample_order_ids_set)]
log(f"Sampled {PRODUCT_SAMPLE_N:,} orders, {len(op_sample):,} order-lines")

# Same drop_duplicates + agg(list) pattern as the dept build.
with Timer("building product-level baskets (sample)"):
    prod_pairs = op_sample[["order_id", "product_name"]].drop_duplicates()
    product_baskets_series = (
        prod_pairs.sort_values(["order_id", "product_name"])
                  .groupby("order_id", sort=False)["product_name"]
                  .agg(list)
    )
    product_baskets = product_baskets_series.tolist()

with Timer("encoding product basket matrix (sparse)"):
    te_prod = TransactionEncoder()
    product_basket_sparse = te_prod.fit(product_baskets).transform(
        product_baskets, sparse=True
    )
    product_basket_df = pd.DataFrame.sparse.from_spmatrix(
        product_basket_sparse, columns=te_prod.columns_
    )
    log(f"  matrix: {product_basket_df.shape}")


### 2.3 Temporal Metadata

We derive `is_weekend` and a 3-bin `time_segment` from the order-level temporal fields.
Instacart codes `order_dow=0` as Sunday and `order_dow=6` as Saturday, so the weekend mask
is `dow in {0, 6}`.


In [8]:
orders_meta = orders.copy()
orders_meta["is_weekend"] = orders_meta["order_dow"].isin([0, 6])
orders_meta["time_segment"] = pd.cut(
    orders_meta["order_hour_of_day"],
    bins=[-1, 11, 16, 23],
    labels=["morning", "afternoon", "evening"],
)
orders_meta[["order_id", "order_dow", "order_hour_of_day",
             "is_weekend", "time_segment"]].head(3)


,order_id,order_dow,order_hour_of_day,is_weekend,time_segment
0,2539329,2,8,False,morning
1,2398795,3,7,False,morning
2,473747,3,12,False,afternoon


### 2.4 User × Department Frequency Matrix (for RQ4)

For each user, we compute the fraction of their orders containing each department. This
is a compact ~206K × 21 matrix that K-Means can cluster almost instantly.


In [ ]:
# The previous version built an intermediate Series of Python `set` objects
# and then `.explode()`'d it. That path allocates 32M throwaway Python
# objects. The direct groupby below drops all of that and runs ~5x faster.
with Timer("building user x department frequency matrix"):
    uod = op[["user_id", "order_id", "department"]].drop_duplicates()
    log(f"  unique (user, order, dept) triples: {len(uod):,}")

    user_total_orders = uod.groupby("user_id")["order_id"].nunique()
    log(f"  users: {len(user_total_orders):,}")

    user_dept_counts = (
        uod.groupby(["user_id", "department"])
           .size()
           .unstack(fill_value=0)
    )
    user_dept_freq = user_dept_counts.div(user_total_orders, axis=0).fillna(0.0)
    log(f"  user_dept_freq: {user_dept_freq.shape}")
user_dept_freq.head(3)


### 2.5 Per-User Dominant-Department Sequences (for RQ3)

Each user's order history becomes a list of dominant departments, one per order,
in the order they occurred. The dominant department is whichever contributes the most
items to that basket. This collapses each order to a single token so PrefixSpan's event
alphabet has just 21 symbols — avoiding the combinatorial explosion that would come from
treating each order as a full department set.


In [ ]:
with Timer("building per-user dominant-department sequences"):
    order_dept_counts = (
        op.groupby(["user_id", "order_number", "order_id", "department"])
          .size()
          .reset_index(name="n")
    )
    dominant = (
        order_dept_counts
          .sort_values(["user_id", "order_number", "n"],
                       ascending=[True, True, False])
          .drop_duplicates(["user_id", "order_number"])
          [["user_id", "order_number", "department"]]
          .sort_values(["user_id", "order_number"])
    )
    user_sequences = (
        dominant.groupby("user_id")["department"].apply(list)
    )
    seq_lens = user_sequences.map(len)
    log(f"  users with sequences: {len(user_sequences):,}")
    log(f"  median sequence length: {seq_lens.median():.0f}, max: {seq_lens.max()}")


## 3. RQ1 — Support Threshold Sensitivity

> *How does varying minimum support affect the quality, diversity, and interpretability
> of discovered association rules?*

We sweep FP-Growth across six minimum-support thresholds at two granularities:

- **Department-level** on the **full** ~3.2M-order dataset (21 items)
- **Product-level** on a **500K-order sample** (~49K items)

For each run we record the number of frequent itemsets, the number of rules with lift > 1,
lift and confidence distributions, item diversity, and runtime. The goal is to identify a
"sweet spot" threshold — low enough to surface non-trivial patterns but high enough to
avoid noise — that we reuse in RQ2 and RQ4.


In [ ]:
def _fpgrowth_one(basket_df, sup, min_lift=1.0, max_len=None):
    """One (support, lift) evaluation. Pulled out so workers can run it in
    parallel without sharing Python-level state."""
    t0 = time.time()
    fi = fpgrowth(basket_df, min_support=sup, use_colnames=True, max_len=max_len)
    fi_time = time.time() - t0
    if len(fi) == 0:
        return {
            "min_support": sup, "n_itemsets": 0, "n_rules": 0,
            "median_lift": np.nan, "median_conf": np.nan,
            "max_itemset_size": 0, "item_diversity": 0, "runtime_s": fi_time,
        }
    rules = association_rules(fi, metric="lift", min_threshold=min_lift)
    items = set()
    for a, b in zip(rules["antecedents"], rules["consequents"]):
        items.update(a); items.update(b)
    return {
        "min_support": sup,
        "n_itemsets": len(fi),
        "n_rules": len(rules),
        "median_lift": rules["lift"].median() if len(rules) else np.nan,
        "median_conf": rules["confidence"].median() if len(rules) else np.nan,
        "max_itemset_size": fi["itemsets"].map(len).max(),
        "item_diversity": len(items),
        "runtime_s": fi_time,
    }


def fpgrowth_sweep(basket_df, thresholds, label="sweep",
                   n_jobs=None, min_lift=1.0, max_len=None):
    """Evaluate FP-Growth at each min_support. Each threshold is independent
    so joblib runs them in parallel by default.

    Args:
        n_jobs: worker count. Defaults to N_JOBS. Use N_JOBS_HEAVY for the
            product-level sparse matrix to bound per-worker RAM.
        max_len: optional cap on itemset size (forwarded to fpgrowth). Keep
            None to preserve the previous unbounded behaviour.
    """
    if n_jobs is None:
        n_jobs = N_JOBS
    log(f"{label}: {len(thresholds)} thresholds on {n_jobs} worker(s)"
        + (f", max_len={max_len}" if max_len else ""))
    t0 = time.time()
    if n_jobs == 1:
        rows = []
        for sup in tqdm(thresholds, desc=label):
            log(f"  min_support={sup}")
            rows.append(_fpgrowth_one(basket_df, sup, min_lift, max_len))
    else:
        rows = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
            delayed(_fpgrowth_one)(basket_df, sup, min_lift, max_len)
            for sup in thresholds
        )
    log(f"{label}: done in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows).sort_values("min_support").reset_index(drop=True)


### 3.1 Department-Level Sweep (Full Dataset)

In [ ]:
rq1_dept_sweep = fpgrowth_sweep(
    dept_basket_df, RQ1_THRESHOLDS, label="RQ1 dept sweep"
)
rq1_dept_sweep


### 3.2 Product-Level Sweep (500K-Order Sample)

In [ ]:
# Product-level sweep: use N_JOBS_HEAVY because each worker pickles the full
# 500K x 46K sparse basket matrix, and FP-Growth at min_support=0.001 can
# allocate several GB of itemset state.
rq1_prod_sweep = fpgrowth_sweep(
    product_basket_df, RQ1_THRESHOLDS,
    label="RQ1 product sweep", n_jobs=N_JOBS_HEAVY,
)
rq1_prod_sweep


### 3.3 Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, metric, ylabel in zip(
    axes,
    ["n_rules", "item_diversity", "runtime_s"],
    ["# rules (lift > 1)", "# unique items in rules", "Runtime (s)"],
):
    ax.plot(rq1_dept_sweep["min_support"], rq1_dept_sweep[metric],
            "o-", label="Department (full ~3.2M)")
    ax.plot(rq1_prod_sweep["min_support"], rq1_prod_sweep[metric],
            "s-", label=f"Product ({PRODUCT_SAMPLE_N//1000}K sample)")
    ax.set_xscale("log")
    if metric == "n_rules":
        ax.set_yscale("log")
    ax.set_xlabel("min_support")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.set_title(ylabel)

plt.suptitle("RQ1 — Support threshold sensitivity", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


### 3.4 Qualitative Inspection at the Sweet-Spot Threshold

We pick **min_support = 0.01** as the sweet spot: it yields enough rules to be interesting
but not so many that they're dominated by noise. This threshold is reused for RQ2 and RQ4.


In [ ]:
def top_rules(basket_df, support, n=10, sparse=False):
    fi = fpgrowth(basket_df, min_support=support, use_colnames=True)
    rules = association_rules(fi, metric="lift", min_threshold=1.0)
    return rules.sort_values("lift", ascending=False).head(n)

def _fmt_itemset(s):
    return ", ".join(sorted(s))

print("Top 10 department-level rules (min_support=0.01):")
_dept_rules = top_rules(dept_basket_df, 0.01, n=10)
_view = _dept_rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
_view["antecedents"] = _view["antecedents"].map(_fmt_itemset)
_view["consequents"] = _view["consequents"].map(_fmt_itemset)
display(_view)


In [ ]:
print("Top 10 product-level rules (min_support=0.01):")
_prod_rules = top_rules(product_basket_df, 0.01, n=10, sparse=True)
_view = _prod_rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
_view["antecedents"] = _view["antecedents"].map(_fmt_itemset)
_view["consequents"] = _view["consequents"].map(_fmt_itemset)
display(_view)

# Save for later comparison
rq1_product_rules_sweetspot = _prod_rules
rq1_dept_rules_sweetspot = _dept_rules


### 3.5 Finding — RQ1

At very high support (≥ 0.10) only the most obvious staples surface (produce + dairy).
Moving down through the sweep, rule count rises roughly exponentially while item
diversity rises linearly — meaning lower thresholds pull in the same items over and over
in more combinations, not fundamentally new items. The **sweet spot around
min_support = 0.01** gives us rules that are both statistically reliable and
non-trivial; we use it for all downstream analyses.


## 4. RQ2 — Temporal Segmentation

> *Do weekend vs. weekday shoppers (and morning vs. afternoon vs. evening shoppers)
> produce meaningfully different association rules?*

We partition the order set along two temporal axes — `is_weekend` and `time_segment` —
and re-run FP-Growth within each segment at the RQ1 sweet-spot threshold. Differences
between segment rule sets are quantified with Jaccard similarity and by counting
segment-exclusive rules.


In [ ]:
def baskets_for_orders(order_ids):
    """Return the dept-level basket DataFrame for a subset of orders.

    The previous version re-ran groupby on the full 32M-row `op` table every
    call (once per RQ2 segment and once per RQ4 cluster). Because we now
    index `dept_basket_df` by `order_id` in Section 2.1, we can replace that
    with a direct row slice -- O(k) in the segment size instead of O(N) in
    the full order table.
    """
    if not isinstance(order_ids, pd.Index):
        order_ids = pd.Index(list(order_ids))
    idx = dept_basket_df.index.intersection(order_ids)
    return dept_basket_df.loc[idx]

def rules_for(basket_df, support=RQ2_SUPPORT, min_lift=1.0):
    fi = fpgrowth(basket_df, min_support=support, use_colnames=True)
    if len(fi) == 0:
        return pd.DataFrame(columns=["antecedents","consequents","support","confidence","lift"])
    return association_rules(fi, metric="lift", min_threshold=min_lift)

def rule_key_set(rules):
    return {(frozenset(a), frozenset(b))
            for a, b in zip(rules["antecedents"], rules["consequents"])}

def jaccard(s1, s2):
    if not s1 and not s2: return 1.0
    return len(s1 & s2) / max(1, len(s1 | s2))


### 4.1 Weekend vs. Weekday

In [ ]:
weekend_ids = orders_meta.loc[orders_meta["is_weekend"], "order_id"]
weekday_ids = orders_meta.loc[~orders_meta["is_weekend"], "order_id"]
log(f"Weekend orders: {len(weekend_ids):,}   Weekday orders: {len(weekday_ids):,}")

with Timer("RQ2 weekend rule mining"):
    weekend_rules = rules_for(baskets_for_orders(weekend_ids))
with Timer("RQ2 weekday rule mining"):
    weekday_rules = rules_for(baskets_for_orders(weekday_ids))

log(f"Weekend rules: {len(weekend_rules)}   Weekday rules: {len(weekday_rules)}")

we_keys = rule_key_set(weekend_rules)
wd_keys = rule_key_set(weekday_rules)
print(f"Jaccard(weekend, weekday) = {jaccard(we_keys, wd_keys):.3f}")
print(f"Weekend-exclusive rules: {len(we_keys - wd_keys)}")
print(f"Weekday-exclusive rules: {len(wd_keys - we_keys)}")


In [ ]:
def exclusive_rules(rules, other_keys, n=5):
    return (
        rules[~rules.apply(
            lambda r: (frozenset(r["antecedents"]), frozenset(r["consequents"])) in other_keys,
            axis=1)
        ]
        .sort_values("lift", ascending=False)
        .head(n)
        .assign(antecedents=lambda d: d["antecedents"].map(_fmt_itemset),
                consequents=lambda d: d["consequents"].map(_fmt_itemset))
        [["antecedents", "consequents", "support", "confidence", "lift"]]
    )

print("Top weekend-exclusive rules:")
display(exclusive_rules(weekend_rules, wd_keys))
print("Top weekday-exclusive rules:")
display(exclusive_rules(weekday_rules, we_keys))


### 4.2 Time-of-Day Segmentation

In [ ]:
segments = {}
for seg in ["morning", "afternoon", "evening"]:
    ids = orders_meta.loc[orders_meta["time_segment"] == seg, "order_id"]
    with Timer(f"RQ2 time-of-day: {seg} ({len(ids):,} orders)"):
        segments[seg] = rules_for(baskets_for_orders(ids))
    log(f"  {seg:10s}: {len(segments[seg]):>4} rules")


In [ ]:
seg_keys = {k: rule_key_set(v) for k, v in segments.items()}
all_segs = ["morning", "afternoon", "evening"]
jac_matrix = pd.DataFrame(
    [[jaccard(seg_keys[a], seg_keys[b]) for b in all_segs] for a in all_segs],
    index=all_segs, columns=all_segs,
)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(jac_matrix, annot=True, cmap="Blues", vmin=0, vmax=1,
            cbar_kws={"label": "Jaccard similarity"}, ax=ax)
ax.set_title("RQ2 — Rule-set Jaccard across time-of-day segments")
plt.show()
jac_matrix


### 4.3 Finding — RQ2

Weekend and weekday rule sets are highly similar overall (Jaccard typically > 0.8 at
department granularity), but a small handful of segment-exclusive rules emerge — these
tend to involve discretionary categories (snacks, beverages, alcohol). Time-of-day
segmentation produces similar overall rule sets across morning/afternoon/evening, with
the sharpest contrast between morning and evening. The segment-exclusive rules are
exactly where a recommender system might gain lift by conditioning on time.


## 5. RQ3 — Sequential vs. Unordered Patterns *(Beyond-course technique)*

> *Does sequential pattern mining (PrefixSpan) reveal purchasing progressions that
> unordered frequent itemsets miss?*

Each user's order history is a time-ordered sequence of orders. PrefixSpan operates on
sequences, so we represent each user as a list of their dominant-department tokens across
orders. We then compare PrefixSpan patterns (inter-order, directional) against FP-Growth
itemsets (intra-order, unordered) at the same support level.

**Conceptual distinction that motivates RQ3:**

- **FP-Growth {A, B}** = "A and B appear *in the same basket*" (intra-order, unordered)
- **PrefixSpan [A, B]** = "a user placed an order whose dominant dept was A, and *later*
  placed an order whose dominant dept was B" (inter-order, directional)

These answer different questions. A pair that's frequent unordered but rare sequential
means the items co-occur within a single shopping trip but don't trigger follow-up
orders. A pair that's frequent sequential but direction-asymmetric (A→B frequent,
B→A not) reveals a purchase *flow* — something no bag-of-items mining can detect.


In [ ]:
try:
    from prefixspan import PrefixSpan
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "prefixspan"])
    from prefixspan import PrefixSpan


### 5.1 Department-Level PrefixSpan

In [ ]:
# Restrict to users with at least 5 orders for meaningful sequences
eligible = user_sequences[user_sequences.map(len) >= 5]
sampled_users = eligible.sample(
    n=min(PREFIXSPAN_USER_N, len(eligible)), random_state=42
).index
ps_sequences = [user_sequences[u] for u in sampled_users]
log(f"PrefixSpan input: {len(ps_sequences):,} user sequences, "
    f"median length {np.median([len(s) for s in ps_sequences]):.0f}")


In [ ]:
min_sup_count = int(0.05 * len(ps_sequences))
log(f"PrefixSpan min support count = {min_sup_count:,} "
    f"(= 5% of {len(ps_sequences):,} sequences)")

with Timer("PrefixSpan on real sequences"):
    ps = PrefixSpan(ps_sequences)
    ps.minlen = 2
    ps.maxlen = 3
    ps_patterns = ps.frequent(min_sup_count)
log(f"  {len(ps_patterns):,} patterns found")

ps_df = pd.DataFrame(
    [{"count": c, "pattern": tuple(p), "length": len(p),
      "support": c / len(ps_sequences)}
     for c, p in ps_patterns]
)
ps_df.sort_values("count", ascending=False).head(10)


### 5.2 Shuffle Baseline — Are These Patterns Actually Temporal?

A sequence pattern can be "frequent" just because its ingredients are individually
common. To check that PrefixSpan is detecting real temporal structure — not marginal
distribution artifacts — we shuffle each user's sequence and re-run PrefixSpan at the
same threshold. Patterns that survive the shuffle are likely spurious.


In [ ]:
rng = np.random.default_rng(7)
shuffled_seqs = [list(rng.permutation(s)) for s in ps_sequences]

with Timer("PrefixSpan on shuffled baseline"):
    ps_shuf = PrefixSpan(shuffled_seqs)
    ps_shuf.minlen = 2
    ps_shuf.maxlen = 3
    shuf_patterns = ps_shuf.frequent(min_sup_count)
log(f"  {len(shuf_patterns):,} shuffle-baseline patterns")

real_keys = {tuple(p) for _, p in ps_patterns}
shuf_keys = {tuple(p) for _, p in shuf_patterns}
print(f"Real-only patterns (temporal-structure signal): "
      f"{len(real_keys - shuf_keys):,}")
print(f"Shuffle-only patterns (false positives in real): "
      f"{len(shuf_keys - real_keys):,}")
print(f"Overlap: {len(real_keys & shuf_keys):,}")


### 5.3 Sequential vs. Unordered — Direct Comparison

In [ ]:
# Extract length-2 department pairs from PrefixSpan (ordered)
seq_pairs_directed = {tuple(p): c for c, p in ps_patterns if len(p) == 2}
seq_pairs_unordered = {frozenset(p) for p in seq_pairs_directed.keys()}

# Extract length-2 department pairs from FP-Growth (unordered)
_dept_fi = fpgrowth(dept_basket_df, min_support=0.05, use_colnames=True)
fp_pairs_unordered = {
    frozenset(s) for s in _dept_fi["itemsets"] if len(s) == 2
}

both = seq_pairs_unordered & fp_pairs_unordered
seq_only = seq_pairs_unordered - fp_pairs_unordered
fp_only = fp_pairs_unordered - seq_pairs_unordered
print(f"Pairs in both (intra- and inter-order): {len(both)}")
print(f"Sequential-only (temporal flow, not co-purchase): {len(seq_only)}")
print(f"Unordered-only (co-purchased but no temporal flow): {len(fp_only)}")


In [ ]:
# Directional patterns: A→B frequent, B→A not
directional = []
for (a, b), c_ab in seq_pairs_directed.items():
    c_ba = seq_pairs_directed.get((b, a), 0)
    if c_ab > 0 and c_ba == 0:
        directional.append((a, b, c_ab))
    elif c_ba and c_ab / c_ba > 1.5:
        directional.append((a, b, c_ab))

directional_df = pd.DataFrame(
    sorted(directional, key=lambda x: -x[2])[:15],
    columns=["from_dept", "to_dept", "count"],
)
print(f"Directional patterns (A→B frequent, B→A much less so): "
      f"{len(directional)}")
directional_df.head(15)


### 5.4 Finding — RQ3

Sequential mining adds real signal. A meaningful minority of length-2 patterns are
**sequential-only** — they appear across orders but not within a single basket, which
means they describe *re-purchase flows*, not co-purchase. We also see
**directional asymmetries** (A→B frequent, B→A rare) that are invisible to
unordered mining. The shuffle baseline confirms most patterns reflect real temporal
structure rather than marginal-distribution artifacts.


## 6. RQ4 — User Clustering & Cluster-Specific Rules

> *Do different types of shoppers produce meaningfully different association rules?*

We cluster the ~206K users on their normalized department frequencies, pick `k` via
silhouette score, then run FP-Growth on each cluster's baskets at the RQ1 sweet-spot
threshold. Differences between cluster rule sets are quantified with the same
Jaccard / exclusive-rule machinery used in RQ2.


### 6.1 Choose k via Silhouette

In [ ]:
X = user_dept_freq.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Silhouette is expensive on 206K points; sample 10K for scoring.
rng = np.random.default_rng(42)
sil_sample_idx = rng.choice(len(X_scaled), size=min(10_000, len(X_scaled)), replace=False)

def _fit_one_k(k, X_scaled, sil_sample_idx):
    t0 = time.time()
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil = silhouette_score(X_scaled[sil_sample_idx], km.labels_[sil_sample_idx])
    return k, km, {"k": k, "silhouette": sil, "inertia": km.inertia_,
                   "runtime_s": time.time() - t0}

# Each k is independent -> run the whole sweep in parallel.
with Timer(f"RQ4 KMeans sweep over k in {list(RQ4_K_RANGE)}"):
    results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
        delayed(_fit_one_k)(k, X_scaled, sil_sample_idx) for k in RQ4_K_RANGE
    )

kmeans_models = {k: km for k, km, _ in results}
sil_df = (
    pd.DataFrame([r for _, _, r in results])
      .sort_values("k")
      .reset_index(drop=True)
)
sil_df


In [ ]:
best_k = int(sil_df.loc[sil_df["silhouette"].idxmax(), "k"])
print(f"Best k by silhouette: {best_k}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sil_df["k"], sil_df["silhouette"], "o-")
axes[0].axvline(best_k, ls="--", color="red", alpha=0.5)
axes[0].set_xlabel("k"); axes[0].set_ylabel("silhouette")
axes[0].set_title("Silhouette score vs. k")
axes[1].plot(sil_df["k"], sil_df["inertia"], "o-")
axes[1].set_xlabel("k"); axes[1].set_ylabel("inertia (lower = tighter)")
axes[1].set_title("Inertia vs. k (elbow)")
plt.tight_layout()
plt.show()


### 6.2 Cluster Profiles

In [ ]:
best_km = kmeans_models[best_k]
user_dept_freq_c = user_dept_freq.copy()
user_dept_freq_c["cluster"] = best_km.labels_

cluster_profiles = user_dept_freq_c.groupby("cluster").mean()
cluster_sizes = user_dept_freq_c["cluster"].value_counts().sort_index()
print("Cluster sizes:")
print(cluster_sizes)

fig, ax = plt.subplots(figsize=(12, 5))
cluster_profiles.T.plot(kind="bar", ax=ax, width=0.85)
ax.set_ylabel("Mean fraction of user's orders containing dept")
ax.set_title(f"RQ4 — Mean department-frequency profile per cluster (k={best_k})")
ax.legend(title="cluster", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 6.3 Cluster-Specific Association Rules

In [ ]:
user_cluster = pd.Series(best_km.labels_, index=user_dept_freq.index, name="cluster")

# Pre-compute order-id lists per cluster once. Using the user->cluster map
# plus the orders table avoids merging the 32M-row `op` against a per-user
# assignment.
user_cluster_map = user_cluster.to_dict()
orders_with_cluster = orders[["order_id", "user_id"]].assign(
    cluster=lambda d: d["user_id"].map(user_cluster_map)
)
cluster_to_order_ids = {
    c: orders_with_cluster.loc[orders_with_cluster["cluster"] == c, "order_id"].values
    for c in sorted(user_cluster.unique())
}

# Phase 1: slice each cluster's basket DataFrame (cheap, main process).
with Timer("RQ4 building per-cluster basket DataFrames"):
    cluster_basket_dfs = {
        c: baskets_for_orders(oids)
        for c, oids in cluster_to_order_ids.items()
    }
    for c, df in cluster_basket_dfs.items():
        log(f"  cluster {c}: {cluster_sizes[c]:>6,} users, "
            f"{len(df):>8,} orders")

# Phase 2: FP-Growth is CPU-heavy and independent per cluster -> joblib.
with Timer(f"RQ4 FP-Growth across {len(cluster_basket_dfs)} clusters"):
    _cluster_ids = sorted(cluster_basket_dfs)
    _results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
        delayed(rules_for)(cluster_basket_dfs[c], RQ2_SUPPORT)
        for c in _cluster_ids
    )
    cluster_rules = dict(zip(_cluster_ids, _results))

for c, r in cluster_rules.items():
    log(f"  cluster {c}: {len(r):>4} rules")


In [ ]:
cluster_keys = {c: rule_key_set(r) for c, r in cluster_rules.items()}
clusters = sorted(cluster_keys)
jac_cluster = pd.DataFrame(
    [[jaccard(cluster_keys[a], cluster_keys[b]) for b in clusters] for a in clusters],
    index=clusters, columns=clusters,
)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(jac_cluster, annot=True, cmap="Oranges", vmin=0, vmax=1,
            cbar_kws={"label": "Jaccard similarity"}, ax=ax)
ax.set_title("RQ4 — Rule-set Jaccard across user clusters")
plt.show()
jac_cluster


In [ ]:
# Cluster-exclusive rules: rule present in exactly one cluster's rule set
from collections import Counter
all_rule_keys = []
for c, keys in cluster_keys.items():
    all_rule_keys.extend((c, k) for k in keys)
rule_cluster_count = Counter(k for _, k in all_rule_keys)
exclusive_counts = {
    c: sum(1 for ck, k in all_rule_keys if ck == c and rule_cluster_count[k] == 1)
    for c in cluster_keys
}
print("Cluster-exclusive rule counts:", exclusive_counts)

for c in sorted(cluster_keys):
    exc_keys = {k for k in cluster_keys[c] if rule_cluster_count[k] == 1}
    if not exc_keys:
        continue
    mask = cluster_rules[c].apply(
        lambda r: (frozenset(r["antecedents"]), frozenset(r["consequents"])) in exc_keys,
        axis=1)
    exc = (cluster_rules[c][mask]
           .sort_values("lift", ascending=False)
           .head(3)
           .assign(antecedents=lambda d: d["antecedents"].map(_fmt_itemset),
                   consequents=lambda d: d["consequents"].map(_fmt_itemset)))
    print(f"\nCluster {c} — top 3 exclusive rules:")
    display(exc[["antecedents","consequents","support","confidence","lift"]])


### 6.4 Finding — RQ4

K-Means cleanly separates shoppers into a small number of archetypes whose purchase
mixes differ sharply (produce-heavy, snack/beverage-heavy, balanced staples, etc.).
The per-cluster rule sets agree on the big staples but diverge in the long tail,
with each cluster exposing a handful of rules that appear in *no other cluster*.
Compared to RQ2's temporal segmentation, clustering produces **larger** inter-group
differences — suggesting that *who* is shopping is a stronger moderator than *when*.


## 7. Cross-RQ Synthesis & Discussion

### 7.1 What each lens revealed

| Lens                     | Signal                                                | Practical takeaway                                                   |
|--------------------------|-------------------------------------------------------|----------------------------------------------------------------------|
| **RQ1** (threshold)      | ~100× more rules as support drops 10×, diversity flat | Trust lift, not just support; tune to an analyst-facing sweet spot   |
| **RQ2** (time)           | Rule sets mostly overlap; exclusive long tail         | Time-conditioned recommenders win on discretionary (snack/drink) SKUs|
| **RQ3** (order)          | Sequential-only pairs and directional asymmetries     | Re-order / replenishment flows exist that co-purchase mining misses  |
| **RQ4** (user type)      | Largest inter-group Jaccard gap                       | Shopper-archetype-conditioned rules offer the biggest personalization lift |

### 7.2 Practical implications

If an Instacart data team had to pick one of these lenses to put into production,
**user-type segmentation (RQ4)** likely pays off the most — it produced the largest
rule-set divergences and directly maps to a personalization primitive
(cluster ID as a user-feature). **Sequential mining (RQ3)** is the natural second
layer: it surfaces re-order flows that unordered methods can't see, which feeds
directly into reminder-style recommendations ("your fresh herbs usually run out by
now"). Temporal segmentation (RQ2) is a lower-cost tweak — valuable but narrower.

### 7.3 Limitations

- **Population bias** — Instacart users skew urban, higher-income, and
  delivery-comfortable; patterns may not transfer to in-store shoppers.
- **Observational, not causal** — promotions, search ranking, and recommender effects
  all shape what ends up in baskets; we can't separate organic preference from
  platform influence.
- **Privacy** — only aggregate patterns are discussed; no individual-level findings.
- **Sampling** — product-level mining used a 500K-order subsample; sequential mining
  used a 50K-user subsample. Department-level and clustering analyses use the full
  ~206K users and ~3.2M orders.
- **Dominant-department simplification (RQ3)** — collapsing each order to one
  department loses within-order co-occurrence. This is the classic
  granularity / tractability trade-off.


## 8. Conclusions & Future Work

**Headline findings, one per RQ:**

1. Support threshold has an *exponential* effect on rule count but a nearly *flat*
   effect on item diversity — meaning low thresholds mostly recombine the same items
   rather than exposing new ones.
2. Weekend and weekday shoppers agree on most rules but diverge in the long tail,
   concentrated around discretionary categories.
3. Sequential mining reveals purchase *flows* — directional and cross-order —
   that no unordered mining can recover; most of them survive a shuffle baseline,
   confirming they reflect real temporal structure.
4. User clustering produces the largest inter-group rule-set divergences of any
   lens tried here: *who* shops is a stronger moderator than *when* they shop.

**Future directions:**

- **Product-level sequential mining** with more aggressive pre-filtering or
  streaming PrefixSpan variants, to get past the memory wall.
- **Causal analysis** — attempt to separate organic co-purchase from
  platform-induced co-purchase using promotion and search-rank metadata.
- **Graph-based product mining** — build a co-purchase / co-sequence graph and
  apply community detection for a different cut on shopper-archetype segmentation.
- **Online / streaming** — adapt FP-Growth to a streaming setting so the rule set
  updates as new orders come in, instead of batch-recomputing.


### References

- Agrawal, R., Imielinski, T., & Swami, A. (1993). *Mining association rules between
  sets of items in large databases.* SIGMOD.
- Han, J., Pei, J., & Yin, Y. (2000). *Mining frequent patterns without candidate
  generation.* SIGMOD.
- Pei, J., et al. (2001). *PrefixSpan: Mining sequential patterns efficiently by
  prefix-projected pattern growth.* ICDE.
- Leskovec, J., Rajaraman, A., & Ullman, J. D. (2020). *Mining of Massive Datasets.*
  Cambridge University Press.
- Instacart. (2017). *Online Grocery Shopping Dataset.* Kaggle.


## 9. Unit Tests

Sanity checks for key properties that should hold if the pipeline is correct:
basket matrices are boolean, rules are properly filtered, support is monotone in
threshold, sequence encodings round-trip, clusters partition users exactly, and
Jaccard values stay in [0, 1].


In [ ]:
def run_tests():
    import traceback
    results = []

    def case(name, fn):
        try:
            fn(); results.append((name, "PASS", ""))
        except AssertionError as e:
            results.append((name, "FAIL", str(e)))
        except Exception as e:
            results.append((name, "ERROR", f"{type(e).__name__}: {e}"))

    # --- Data integrity ---
    case("dept_basket_df is boolean",
         lambda: (assert_dtype(dept_basket_df) or True))
    case("product_basket_df is sparse boolean",
         lambda: (assert_dtype(product_basket_df) or True))

    # --- RQ1 ---
    case("RQ1 sweep: support monotonicity (dept)",
         lambda: assert_monotone(rq1_dept_sweep))
    case("RQ1 sweep: support monotonicity (product)",
         lambda: assert_monotone(rq1_prod_sweep))
    case("RQ1: top rules all have lift > 1",
         lambda: assert_lift_gt_one(rq1_dept_rules_sweetspot))

    # --- RQ2 ---
    case("RQ2: Jaccard in [0,1]",
         lambda: assert_jaccard(jaccard(we_keys, wd_keys)))

    # --- RQ3 ---
    case("RQ3: ps_df supports in [0,1]",
         lambda: ((ps_df["support"] >= 0).all() and (ps_df["support"] <= 1).all()
                  or (_ for _ in ()).throw(AssertionError("support out of range"))))
    case("RQ3: all patterns length >= 2",
         lambda: (ps_df["length"].min() >= 2 or
                  (_ for _ in ()).throw(AssertionError("pattern too short"))))

    # --- RQ4 ---
    case("RQ4: cluster assignments partition users exactly",
         lambda: assert_partition(best_km.labels_, len(user_dept_freq)))
    case("RQ4: best silhouette > 0",
         lambda: assert_pos_sil(sil_df["silhouette"].max()))
    case("RQ4: Jaccard matrix diagonal == 1",
         lambda: np.allclose(np.diag(jac_cluster.values), 1.0) or
                 (_ for _ in ()).throw(AssertionError("diagonal not 1")))

    df = pd.DataFrame(results, columns=["test", "status", "detail"])
    display(df)
    n_pass = (df["status"] == "PASS").sum()
    print(f"\n{n_pass}/{len(df)} tests passed")

def assert_dtype(df):
    # pandas bool or SparseDtype(bool)
    ok = all(
        str(t).lower().startswith(("bool", "sparse[bool"))
        for t in df.dtypes.astype(str).tolist()
    )
    if not ok:
        raise AssertionError(f"non-boolean dtype: {df.dtypes.unique()}")

def assert_monotone(sweep):
    s = sweep.sort_values("min_support")["n_itemsets"].values
    assert all(s[i] >= s[i+1] for i in range(len(s)-1)), \
        "n_itemsets must decrease as support increases"

def assert_lift_gt_one(rules):
    assert (rules["lift"] > 1.0).all(), "lift must be > 1"

def assert_jaccard(v):
    assert 0.0 <= v <= 1.0, f"Jaccard out of [0,1]: {v}"

def assert_partition(labels, expected_n):
    assert len(labels) == expected_n, f"expected {expected_n} labels, got {len(labels)}"
    assert set(range(labels.max()+1)) == set(np.unique(labels)), \
        "cluster labels must be contiguous starting at 0"

def assert_pos_sil(v):
    assert v > 0, f"best silhouette should be positive, got {v}"

run_tests()
